# Batch collection and sentiment

Original coursework with security and small integration fixes. Outputs cleared. Requires your own Reddit API access; running makes live API requests. See README.md and the root EDITS.md.


In [ ]:
import praw
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
import os

client_id = os.environ["REDDIT_CLIENT_ID"]
client_secret = os.environ["REDDIT_CLIENT_SECRET"]
user_agent = os.environ["REDDIT_USER_AGENT"]

# Initialise PRAW
reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent
)

In [ ]:
def fetch_reddit_data(my_subreddits, keywords, limit=2000):
    """
    Fetches posts and comments containing specific keywords from chosen subreddits.
    """
    data = []

    for i in range(len(my_subreddits)):
    
        subreddit = reddit.subreddit(my_subreddits[i])
        
        for submission in subreddit.new(limit=limit):
            # Check if the post contains any keywords
            post_keywords = [keyword for keyword in keywords if keyword.lower() in (submission.title + submission.selftext).lower()]
            
            if post_keywords:
                # Add main post data
                post_data = {
                    'type': 'post',
                    'content': submission.title + " " + submission.selftext,
                    'author': submission.author.name if submission.author else 'N/A',
                    'score': submission.score,
                    'created_utc': int(submission.created_utc),
                    'human_readable_time': datetime.fromtimestamp(int(submission.created_utc)).strftime('%d-%m-%Y %H:%M:%S'),
                    'url': f"https://www.reddit.com{submission.permalink}",
                    'keywords': ', '.join(post_keywords),
                }
                data.append(post_data)
    
                # Process comments
                submission.comments.replace_more(limit=None)  # Expand all comments
                for comment in submission.comments.list():
                    # Check if the comment contains any keywords
                    comment_keywords = [keyword for keyword in keywords if keyword.lower() in comment.body.lower()]
                    if comment_keywords:
                        comment_data = {
                            'type': 'comment',
                            'content': comment.body,
                            'author': comment.author.name if comment.author else 'N/A',
                            'score': comment.score,
                            'created_utc': int(comment.created_utc),
                            'human_readable_time': datetime.fromtimestamp(int(comment.created_utc)).strftime('%d-%m-%Y %H:%M:%S'),
                            'url': f"https://www.reddit.com{submission.permalink}",
                            'keywords': ', '.join(comment_keywords),
                        }
                        data.append(comment_data)
    
    return pd.DataFrame(data)

In [ ]:
def num_keywords(keywords):
    """
    Splits the keywords into individual strings
    """
    return len(keywords.split(", "))

def add_keyword_columns(df):
    """
    Adds the number of keyword columns for the maximum number of keywords found in any post or comment
    """
    all_keywords = []
    df["keyword count"] = df.loc[:,"keywords"].apply(num_keywords)
    
    for i in range(df.loc[:,"keywords"].apply(num_keywords).max()):
            df[f"keyword {i+1}"] = ""
        
    for i in range(len(df.loc[:,"keywords"])):
        all_keywords.append((df.loc[i,"keywords"].split(", ")))
    
    for i in range(len(all_keywords)):
        for j in range(len(all_keywords[i])):
            df.iloc[i,j+9] = all_keywords[i][j]
            
    return (all_keywords)

In [ ]:
# Choosing subreddits and keywords I am searching for

my_subreddits_eco = ["news","worldnews","climate","geography","weather"]

keywords_eco = ["extreme weather", "severe weather", "weather emergency", 
 "natural disaster", "hurricane", "cyclone", 
 "typhoon", "flood", "wildfire", "disaster relief"]

reddit_data_eco = fetch_reddit_data(my_subreddits_eco,keywords_eco) # Fetch data from Reddit

time = datetime.now().strftime("%d-%m-%Y_%H-%M-%S")
reddit_data_eco.to_csv(f'reddit_data_eco_{time}.csv', index=False) # Save data with the current time

In [ ]:
all_keywords = add_keyword_columns(reddit_data_eco) # Add the separate keyword columns
reddit_data_eco.to_csv('clean_eco.csv',index=False) # Save data with added keyword columns

In [ ]:
from textblob import TextBlob

def analyse_sentiment(text):
    """
    Returns sentiment polarity for a string
    """
    analysis = TextBlob(text)
    return analysis.sentiment.polarity

In [ ]:
text = reddit_data_eco["content"]

# Analyse sentiment for each post/comment
sentiments = [analyse_sentiment(content) for content in text]

sentiment_text = pd.DataFrame({'type': reddit_data_eco['type'], 'content': text, 'polarity': sentiments, 'keywords': reddit_data_eco['keywords'], 'url': reddit_data_eco['url']})

# Convert polarity to sentiment
sentiment_text['sentiment'] = sentiment_text['polarity'].apply(lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'neutral'))
sentiment_text.to_csv('sentiment_and_text_eco.csv', index=False)

In [ ]:
import random
def random_example_posts(df, num=3, seed=42):
    """
    returns a specified number of random posts from the dataframe
    using a seed so that it is repeatable
    """
    random.seed(seed)
    rand_indices = random.sample(range(len(df)), num)
    return df.iloc[rand_indices][["type","content", "sentiment","url"]]

In [ ]:
random_posts = random_example_posts(sentiment_text)
random_posts.to_csv('random_posts.csv', index=False)